# No.1 FID信号生成とガウスノイズ付加

NMRのFID（Free Induction Decay）信号をシミュレートし、ガウスノイズを付加します。

$$s(t) = a_1 e^{-t/T_{1,1}} + a_2 e^{-t/T_{1,2}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import scipy.io


## パラメータ設定

In [ ]:
a1, a2 = 1.0, 1.0       # 振幅
t1_1, t1_2 = 2.0, 0.5   # 緩和時間 [s]
dt = 0.02                # サンプリング間隔 [s]
sigma = 0.03             # ガウスノイズ標準偏差
DNUM = 1024              # データ点数

t = np.arange(DNUM) * dt
print(f"計測時間: 0 〜 {t[-1]:.2f} s")

## FID信号生成（指数減衰2成分）

In [ ]:
Signal = np.zeros(DNUM)

for k in range(DNUM):
    v1 = -t[k] / t1_1
    v2 = -t[k] / t1_2
    # e^(-50) 以下は 0 として打ち切る（MATLABと同じ条件）
    s1 = a1 * np.exp(v1) if v1 > -50 else 0.0
    s2 = a2 * np.exp(v2) if v2 > -50 else 0.0
    Signal[k] = s1 + s2

# ノイズなしで確認
plt.figure(figsize=(10, 3))
plt.plot(t, Signal)
plt.title("FID Signal (no noise)")
plt.xlabel("Time [s]")
plt.grid(True)
plt.show()

## ガウスノイズ付加（Box-Muller法）

Box-Muller変換により一様乱数からガウス分布の乱数を生成します：

$$n = \sigma \sqrt{-2 \ln u_1} \cos(2\pi u_2)$$

In [ ]:
rng = np.random.default_rng(seed=42)
u1 = rng.uniform(0, 1, DNUM)
u2 = rng.uniform(0, 1, DNUM)
noise = sigma * np.sqrt(-2 * np.log(u1)) * np.cos(2 * np.pi * u2)

Signal = Signal + noise

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, Signal, linewidth=0.8)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Signal Intensity")
ax.set_title("FID Signal with Gaussian Noise")
ax.grid(True)
fig.tight_layout()
fig.savefig("NoisySignal.png", dpi=150)
plt.show()
print("NoisySignal.png を保存しました")

## データ保存（MATLAB互換）

In [ ]:
# .mat 形式で保存 → MATLABやNo.2以降のFFTでそのまま使える
scipy.io.savemat("Signal.mat", {"Signal": Signal})
print("Signal.mat を保存しました")

# 確認: MATLABで load Signal.mat すると Signal 変数が読み込まれる